# Command Line Examples

## Get Interface

In [ ]:
import json
import threading
import urllib.parse
from http.server import BaseHTTPRequestHandler, HTTPServer

from _fake_data import create_pb_bytes

# PV names returned by the mocked getMatchingPVs (search) endpoint.
_SEARCH_RESULTS = ["EXAMPLE:TEMPERATURE", "EXAMPLE:TEMPERATURE2", "EXAMPLE:PRESSURE"]


class _MockHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        parsed = urllib.parse.urlparse(self.path)
        qs = urllib.parse.parse_qs(parsed.query)
        if "getMatchingPVs" in parsed.path:
            # Search endpoint: return a JSON list of PV names.
            body = json.dumps(_SEARCH_RESULTS).encode()
            content_type = "application/json"
        else:
            # Data endpoint: return PB-encoded events for the requested PV.
            pv = qs.get("pv", ["unknown"])[0]
            body = create_pb_bytes(pv)
            content_type = "application/octet-stream"
        self.send_response(200)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *args):
        pass  # silence server logs


class _ReusableHTTPServer(HTTPServer):
    allow_reuse_address = True


_mock_server = _ReusableHTTPServer(("localhost", 17668), _MockHandler)
_server_thread = threading.Thread(target=_mock_server.serve_forever, daemon=True)
_server_thread.start()


In [ ]:
!arch-retrieval get --help

In [ ]:
!arch-retrieval --hostname localhost get EXAMPLE:TEMPERATURE

In [ ]:
!arch-retrieval --hostname localhost get EXAMPLE:TEMPERATURE -p min -b 5

In [ ]:
!arch-retrieval --hostname localhost get EXAMPLE:TEMPERATURE -p ncount

In [ ]:
!arch-retrieval --hostname localhost get EXAMPLE:TEMPERATURE2 EXAMPLE:TEMPERATURE3  -s 2024-12-01T12:00:00 -e 2024-12-16T12:10:10.10 -p MEDIAN -b 6000

## Export Data

The `export` command writes a single PV's data to stdout in a machine-readable format: `json` (default), `csv`, `arrow`, `parquet`, or `pb` (the raw Archiver Appliance protobuf). The `csv`/`arrow`/`parquet` formats require the `[polars]` extra. Redirect stdout to a file to save the result.

In [ ]:
!arch-retrieval export --help

In [ ]:
!arch-retrieval --hostname localhost export --format csv EXAMPLE:TEMPERATURE | head

In [ ]:
!arch-retrieval --hostname localhost export --format pb EXAMPLE:TEMPERATURE > example.pb

## Search for PV Names

The `search` command looks up PV names matching a regex pattern. Optionally pass `-s`/`-e` to only return PVs that recorded data in a time range, and `-l` to change the result limit.

In [ ]:
!arch-retrieval search --help

In [ ]:
!arch-retrieval --hostname localhost search "EXAMPLE:.*"

## Read a Local PB File

The `read-pb` command displays the events stored in a local Archiver Appliance `.pb` file (for example one written by `export --format pb` above) as a table, without contacting a server.

In [ ]:
!arch-retrieval read-pb --help

In [ ]:
!arch-retrieval --hostname localhost read-pb example.pb

In [ ]:
import os

_mock_server.shutdown()
if os.path.exists("example.pb"):
    os.remove("example.pb")